# 可选实验：特征工程和多项式回归

![](./images/C1_W2_Lab07_FeatureEngLecture.PNG)


## 目标
在本实验中，您将：
- 探索特征工程和多项式回归，它允许您使用线性回归的机制来拟合非常复杂、甚至非常非线性的函数。


## 工具
您将使用之前实验中开发的函数以及matplotlib和NumPy。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lab_utils_multi import zscore_normalize_features, run_gradient_descent_feng
np.set_printoptions(precision=2)  # 减少numpy数组的显示精度

<a name='FeatureEng'></a>
# 特征工程和多项式回归概述

开箱即用，线性回归提供了一种构建以下形式模型的方法：
$$
f_{\mathbf{w},b} = w_0x_0 + w_1x_1+ ... + w_{n-1}x_{n-1} + b \tag{1}
$$ 
如果您的特征/数据是非线性的或特征的组合怎么办？例如，房价往往不与居住面积呈线性关系，而是对非常小或非常大的房屋进行惩罚，导致上图中显示的曲线。我们如何使用线性回归的机制来拟合这条曲线？请记住，我们拥有的'机制'是能够修改(1)中的参数$\mathbf{w}$，$\mathbf{b}$来'拟合'方程到训练数据。然而，无论怎样调整(1)中的$\mathbf{w}$，$\mathbf{b}$都无法实现对非线性曲线的拟合。


<a name='PolynomialFeatures'></a>
## 多项式特征

上面我们考虑的是数据非线性的场景。让我们尝试使用我们目前所知来拟合非线性曲线。我们将从一个简单的二次方程开始：$y = 1+x^2$

您熟悉我们使用的所有例程。它们可以在lab_utils.py文件中查看。我们将使用[`np.c_[..]`](https://numpy.org/doc/stable/reference/generated/numpy.c_.html)，这是一个沿列边界连接的NumPy例程。

In [ ]:
# 创建目标数据
x = np.arange(0, 20, 1)
y = 1 + x**2
X = x.reshape(-1, 1)

model_w,model_b = run_gradient_descent_feng(X,y,iterations=1000, alpha = 1e-2)

plt.scatter(x, y, marker='x', c='r', label="实际值"); plt.title("无特征工程")
plt.plot(x,X@model_w + model_b, label="预测值");  plt.xlabel("X"); plt.ylabel("y"); plt.legend(); plt.show()

嗯，正如预期的那样，拟合效果不太好。需要的是类似$y= w_0x_0^2 + b$的东西，或者**多项式特征**。
为了实现这一点，您可以修改*输入数据*来*工程化*所需的特征。如果您将原始数据替换为对$x$值进行平方的版本，那么您可以实现$y= w_0x_0^2 + b$。让我们试试。在下面将`X`替换为`X**2`：

In [ ]:
# 创建目标数据
x = np.arange(0, 20, 1)
y = 1 + x**2

# 工程化特征 
X = x**2      #<-- 添加工程化特征

In [ ]:
X = X.reshape(-1, 1)  # X应该是2-D矩阵
model_w,model_b = run_gradient_descent_feng(X, y, iterations=10000, alpha = 1e-5)

plt.scatter(x, y, marker='x', c='r', label="实际值"); plt.title("添加x**2特征")
plt.plot(x, np.dot(X,model_w) + model_b, label="预测值"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

太好了！几乎完美拟合。注意图上方打印的$\mathbf{w}$和b的值：`w,b found by gradient descent: w: [1.], b: 0.0490`。梯度下降将我们初始的$\mathbf{w},b $值修改为(1.0,0.049)，即$y=1*x_0^2+0.049$的模型，非常接近我们的目标$y=1*x_0^2+1$。如果您运行更长时间，它可能会更好地匹配。

### 选择特征
<a name='GDF'></a>
上面，我们知道需要$x^2$项。哪些特征是必需的可能并不总是显而易见的。可以添加各种潜在特征来尝试找到最有用的特征。例如，如果我们尝试：$y=w_0x_0 + w_1x_1^2 + w_2x_2^3+b$会怎样？

运行下一个单元格。

In [ ]:
# 创建目标数据
x = np.arange(0, 20, 1)
y = x**2

# 工程化特征
X = np.c_[x, x**2, x**3]   #<-- 添加工程化特征

In [ ]:
model_w,model_b = run_gradient_descent_feng(X, y, iterations=10000, alpha=1e-7)

plt.scatter(x, y, marker='x', c='r', label="实际值"); plt.title("x, x**2, x**3特征")
plt.plot(x, X@model_w + model_b, label="预测值"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

注意$\mathbf{w}$的值，`[0.08 0.54 0.03]`，b是`0.0106`。这意味着拟合/训练后的模型是：
$$
 0.08x + 0.54x^2 + 0.03x^3 + 0.0106 
$$
梯度下降通过增加$w_1$项相对于其他项来强调最适合$x^2$数据的数据。如果您运行很长时间，它将继续减少其他项的影响。
> 梯度下降通过强调其相关参数为我们选择'正确'的特征

让我们回顾一下这个想法：
- 最初，特征被重新缩放，以便它们可以相互比较
- 较小的权重值意味着不太重要/正确的特征，在极端情况下，当权重变为零或非常接近零时，相关特征对于拟合模型没有用处。
- 上面，拟合后，与$x^2$特征相关的权重远大于$x$或$x^3$的权重，因为它在拟合数据中最有用。

### 替代视图
上面，多项式特征是根据它们与目标数据的匹配程度来选择的。另一种思考方式是，一旦我们创建了新特征，我们仍然在使用线性回归。鉴于此，最好的特征将与目标呈线性关系。这最好通过一个例子来理解。

In [ ]:
# 创建目标数据
x = np.arange(0, 20, 1)
y = x**2

# 工程化特征
X = np.c_[x, x**2, x**3]   #<-- 添加工程化特征
X_features = ['x','x^2','x^3']

In [ ]:
fig,ax=plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for i in range(len(ax)):
    ax[i].scatter(X[:,i],y)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("y")
plt.show()

上面，很明显$x^2$特征与目标值$y$的映射是线性的。然后线性回归可以轻松地使用该特征生成模型。

### 特征缩放
如上一个实验所述，如果数据集的特征尺度差异显著，应应用特征缩放以加速梯度下降。在上面的例子中，有$x$、$x^2$和$x^3$，它们自然会有非常不同的尺度。让我们对我们的示例应用Z-score标准化。

In [ ]:
# 创建目标数据
x = np.arange(0,20,1)
X = np.c_[x, x**2, x**3]
print(f"原始X按列的峰峰值范围:{np.ptp(X,axis=0)}")

# 添加均值归一化 
X = zscore_normalize_features(X)     
print(f"标准化后X按列的峰峰值范围:{np.ptp(X,axis=0)}")

现在我们可以用更激进的alpha值再试一次：

In [ ]:
x = np.arange(0,20,1)
y = x**2

X = np.c_[x, x**2, x**3]
X = zscore_normalize_features(X) 

model_w, model_b = run_gradient_descent_feng(X, y, iterations=100000, alpha=1e-1)

plt.scatter(x, y, marker='x', c='r', label="实际值"); plt.title("标准化x x**2, x**3特征")
plt.plot(x,X@model_w + model_b, label="预测值"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

特征缩放使其收敛得更快。  
再次注意$\mathbf{w}$的值。$w_1$项，即$x^2$项是最被强调的。梯度下降几乎消除了$x^3$项。

### 复杂函数
通过特征工程，甚至可以建模相当复杂的函数：

In [ ]:
x = np.arange(0,20,1)
y = np.cos(x/2)

X = np.c_[x, x**2, x**3,x**4, x**5, x**6, x**7, x**8, x**9, x**10, x**11, x**12, x**13]
X = zscore_normalize_features(X) 

model_w,model_b = run_gradient_descent_feng(X, y, iterations=1000000, alpha = 1e-1)

plt.scatter(x, y, marker='x', c='r', label="实际值"); plt.title("标准化x x**2, x**3特征")
plt.plot(x,X@model_w + model_b, label="预测值"); plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()



## 恭喜！
在本实验中，您：
- 学习了线性回归如何使用特征工程建模复杂、甚至高度非线性的函数
- 认识到在进行特征工程时应用特征缩放的重要性